<a href="https://colab.research.google.com/github/lasigeBioTM/data-text-processing-notebooks/blob/main/notebooks/01-unix-shell.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Unix Shell Tutorial

## Introduction

This is the **first tutorial** in a series that will demonstrate how shell scripting can be used to perform the tasks that health and life science specialists may need to undertake to find and retrieve biomedical data and text. We will use the compound caffeine as an example and explore different public repositories to identify diseases related to it. The focus is not on the specific relationships we may discover, but on the process of obtaining them.

**The objective of this tutorial is to learn how to create and use data and script files in a Unix shell.**

> This tutorial is part of a series of tutorials adapted as interactive versions of the hands-on steps described in the [Data and Text Processing for Health and Life Sciences](https://labs.rd.ciencias.ulisboa.pt/book/) book, which is licensed under the [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).

---

# Step 1: Introduction to the Unix Shell

A shell is a software program that interprets and executes command lines given by the user in consecutive lines of text. A shell script is a list of such command lines. The command line usually starts by invoking a command-line tool.

The Unix shell was developed to manage Unix-like operating systems, but due to its usefulness, it is now available on most personal computers running Linux, macOS, or Windows.

## Current Directory

As our first command line, we can show the full path of the directory (folder) in which the shell is working. The `pwd` command prints the current working directory.

In [1]:
%%bash
pwd

/content


To understand a command-line tool, you can use the `help` command before the command name. For example, to have a more concise description of `pwd`:

In [2]:
%%bash
help pwd

pwd: pwd [-LP]
    Print the name of the current working directory.
    
    Options:
      -L	print the value of $PWD if it names the current working
    		directory
      -P	print the physical directory, without any symbolic links
    
    By default, `pwd' behaves as if `-L' were specified.
    
    Exit Status:
    Returns 0 unless an invalid option is given or the current directory
    cannot be read.


## List Files

The `ls` command will show the list of files in the current directory:

In [3]:
%%bash
ls

sample_data


## Change Directory

To change the directory, we use the `cd` (change directory) command followed by the new path. Let's first try to navigate to a non-existent `Documents` directory:

In [4]:
%%bash
cd Documents || echo "No directory found"

No directory found


bash: line 1: cd: Documents: No such file or directory


**Note**: The previous error is intentional and demonstrates what happens when you try to change into a directory that does not exist.

The `||` operator is used to print a message (or execute a fallback command) in case the preceding command fails.

To create the directory, we use the `mkdir` command:


In [5]:
%%bash
mkdir Documents

Now we can navigate into the Documents directory:

In [6]:
%%bash
cd Documents
pwd

/content/Documents


To return to the parent directory, we use two dots (`..`):

In [7]:
%%bash
cd ..
pwd

/


To return to the home directory directly, we use the tilde character (`~`):

In [8]:
%%bash
cd ~
pwd

/root


## Useful Key Combinations

- **Ctrl-C**: Cancels the current tool being executed. Try using the `cd` command with only one single quote to see how it blocks:
  ```bash
  cd '
  ```
  Press **Ctrl-C** to cancel it.

- **Ctrl-D**: Indicates to the terminal that it is the end of input. Instead of canceling, this executes the command (which will likely result in a syntax error).

- **Ctrl-L**: Clears the terminal display.

- **Ctrl-Insert** and **Shift-Insert**: Copy and paste selected text, respectively.

## Shell Version

The following examples will probably work in any Unix shell, but if we want to be certain that we are using bash, we can check the process running in our terminal:

In [9]:
%%bash
ps | grep $$

   1056 ?        00:00:00 bash


**Expected Output**: Should show information about the Bash process running in this cell.

---

# Step 2: Creating and Using Data and Script Files

Now that we know how to use a shell, let's create our first data file and a script that processes it.

## Create Data File

We'll create a file named `myfile.txt` with simple content. In Jupyter, we can create files directly using bash:

In [10]:
%%bash
cat > myfile.txt << 'EOF'
line 1
line 2
line 3
line 4
EOF

**`<< 'EOF'`**: tells the terminal to start capturing all subsequent lines of text until it reaches the specific keyword `EOF` (End Of File).

## Verify File Contents

Let's check if the file was created properly. The `cat` command displays the file contents:

In [11]:
%%bash
cat myfile.txt

line 1
line 2
line 3
line 4


Alternatively, if you're using a local terminal, you can use a command-line
editor like `nano` by running `nano myfile.txt`, or simply use a graphical
editor like Notepad.



## Reverse File Contents

The `tac` command ("cat" backwards) reverses the order of lines:


In [12]:
%%bash
tac myfile.txt

line 4
line 3
line 2
line 1


## Create Your First Script

Now let's create a script file named `reversemyfile.sh` that reverses the contents of a file. The script will use the `tac` command and receive a filename as its first argument (`$1`):

In [13]:
%%bash
cat > reversemyfile.sh << 'EOF'
tac $1
EOF

In this script:
- `tac` is the command that reverses lines
- `$1` represents the first argument (filename) passed to the script when we invoke it

We could also add the shebang `#!/bin/bash` as the first line to explicitly specify that it should be executed using the Bash shell. However, for simplicity we will not use any shebang in this tutorial.

Let's verify the script was created properly:

In [14]:
%%bash
cat reversemyfile.sh

tac $1


## File Permissions

A script needs execute permission to run. We use the `chmod` command to grant the user (`u`) execute permission (`+x`):

In [15]:
%%bash
chmod u+x reversemyfile.sh

## Execute the Script

Now we can execute the script by providing `myfile.txt` as an argument. The `./` prefix indicates that the script is in the current directory:

In [16]:
%%bash
./reversemyfile.sh myfile.txt

line 4
line 3
line 2
line 1


**Congratulations!** You've made your first script work!

## Scripts with Multiple Arguments

If we provide more arguments, only the first one is used by our script (since it only references `$1`). The others are ignored:

In [17]:
%%bash
./reversemyfile.sh myfile.txt myotherfile.txt 'myother file.txt'

line 4
line 3
line 2
line 1


The output is exactly the same because our script only uses `$1`. The `$2` and `$3` variables would represent the second and third arguments respectively. Note: when arguments contain spaces, they must be enclosed in single quotes.

---

# Step 3: Working with File Redirection and Output

In this final step, we'll learn about Unix file redirection operators and how to save script output to files.

## Line Break Considerations

A Unix file represents a single line break using a **line feed** character, while Windows uses two characters (carriage return and line feed). When working on Windows, use a text editor (for example, Notepad++) that can save in Unix format.

If needed, we can remove extra carriage returns using the `tr` (translate) command:

In [18]:
%%bash
# Create a backup and clean line breaks (this is just for demonstration)
tr -d '\r' < reversemyfile.sh > reversemyfilenew.sh
echo "Cleaned file created"

Cleaned file created


## Redirection Operators

- **`>`** redirects output TO a file (standard output redirection)
- **`<`** redirects output FROM a file (standard input redirection)

For example, instead of providing a filename as an argument, `cat` can receive file contents through standard input:


In [19]:
%%bash
cat < myfile.txt

line 1
line 2
line 3
line 4


The `tr` command couldn't write to the same file it was reading, so we created a new file. Let's keep the cleaned version by moving it:

In [20]:
%%bash
mv reversemyfilenew.sh reversemyfile.sh
chmod u+x reversemyfile.sh

## Save Script Output to a File

We can now save the reversed output to a new file using the `>` redirection operator:

In [21]:
%%bash
./reversemyfile.sh myfile.txt > mynewfile.txt

Let's verify the file was created with the correct contents:


In [22]:
%%bash
cat mynewfile.txt

line 4
line 3
line 2
line 1


If we reverse the reversed file, we should get back the original content:


In [23]:
%%bash
./reversemyfile.sh mynewfile.txt

line 1
line 2
line 3
line 4


## Debugging Scripts

If something is not working correctly, we can debug the script using the `-x` flag with bash. This will display each command executed (preceded by `+`) along with the output:

In [24]:
%%bash
bash -x reversemyfile.sh myfile.txt

line 4
line 3
line 2
line 1


+ tac myfile.txt


Alternatively, you can add `set -x` in your script to start debugging mode, and `set +x` to stop it:

```bash
#!/bin/bash
set -x          # Start debugging
tac $1
set +x          # Stop debugging
```

---

# Exercise: Challenge Yourself!

**Challenge**: Create a script `mirror.sh` that receives as input the name of a file and outputs the contents of that file plus the contents in reverse order.

This means the output of executing the script should contain **double the number of lines** of the input file.

**Hint**: You'll need to use both `cat` and `tac` commands, and combine their outputs. You might want to use the pipe operator (`|`) or redirect the output of one command to another.

Try implementing it below!

In [25]:
%%bash
# TODO: Create mirror.sh script here
# Hint: Consider using cat and tac with appropriate redirection

Once you've created `mirror.sh`, test it here:

**Expected Output** (8 lines total - the original 4 lines plus 4 reversed):
```
line 1
line 2
line 3
line 4
line 4
line 3
line 2
line 1
```

In [26]:
%%bash
# Test your mirror.sh script (once created)
# ./mirror.sh myfile.txt

## Solution

Here's one way to implement the `mirror.sh` script using command substitution or pipes:

In [27]:
%%bash
cat > mirror.sh << 'EOF'
cat $1
tac $1
EOF
chmod u+x mirror.sh
echo "mirror.sh created and made executable"

mirror.sh created and made executable


In [28]:
%%bash
echo "Testing mirror.sh:"
./mirror.sh myfile.txt

Testing mirror.sh:
line 1
line 2
line 3
line 4
line 4
line 3
line 2
line 1


---

# Conclusion

In this tutorial, we learned:

1. **Basic Shell Navigation**
   - `pwd` - Print working directory
   - `ls` - List files
   - `cd` - Change directory
   - `mkdir` - Create directories

2. **File Operations**
   - `cat` - Display file contents
   - `tac` - Display file contents in reverse order
   - `cat >` - Create files

3. **Creating and Executing Scripts**
   - Creating shell script files
   - Setting execute permissions with `chmod`
   - Using positional parameters (`$1`, `$2`, etc.)
   - Running scripts with the `./` prefix

4. **File Redirection**
   - `>` - Redirect output to a file
   - `<` - Redirect input from a file
   - Using `mv` to move files

5. **Debugging**
   - Using `bash -x` for debugging scripts
   - Understanding command execution flow

This concludes the **Unix Shell** tutorial adapted from the [Data and Text Processing for Health and Life Sciences](https://labs.rd.ciencias.ulisboa.pt/book/) book.

The next tutorial in this series will use the compound caffeine as an example and navigate through different public repositories to find diseases related to it.

Happy shell scripting!